In [3]:
# Naive Bayes Text Classification using 20 Newsgroups Dataset
# Comparing Multinomial Naive Bayes and Bernoulli Naive Bayes

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ---------------------------------------------------------
# 1. Load the 20 Newsgroups Dataset
# ---------------------------------------------------------

# Load training data
train_data = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

# Load test data
test_data = fetch_20newsgroups(
    subset="test",
    remove=("headers", "footers", "quotes")
)

X_train_text = train_data.data
y_train = train_data.target

X_test_text = test_data.data
y_test = test_data.target

print("Number of training documents:", len(X_train_text))
print("Number of testing documents:", len(X_test_text))
print("Number of categories:", len(train_data.target_names))

print("\nCategories:")
for i, category in enumerate(train_data.target_names):
    print(i, ":", category)


# ---------------------------------------------------------
# 2. Text Preprocessing and Feature Extraction
# ---------------------------------------------------------

# CountVectorizer converts documents into a matrix of word counts.
#
# stop_words="english" removes common English words.
# min_df=2 ignores words appearing in only one document.
# max_df=0.95 ignores extremely common words.
# ngram_range=(1, 2) uses both unigrams and bigrams.

vectorizer = CountVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

# Fit vectorizer only on training data
X_train = vectorizer.fit_transform(X_train_text)

# Transform test data using the same vocabulary
X_test = vectorizer.transform(X_test_text)

print("\nFeature matrix shape:")
print("Training:", X_train.shape)
print("Testing :", X_test.shape)


# ---------------------------------------------------------
# 3. Multinomial Naive Bayes
# ---------------------------------------------------------

multinomial_nb = MultinomialNB(alpha=1.0)

# Train model
multinomial_nb.fit(X_train, y_train)

# Predict test data
y_pred_mnb = multinomial_nb.predict(X_test)

# Evaluate
mnb_accuracy = accuracy_score(y_test, y_pred_mnb)
mnb_f1 = f1_score(
    y_test,
    y_pred_mnb,
    average="weighted"
)

print("\n========================================")
print("Multinomial Naive Bayes")
print("========================================")
print("Accuracy :", mnb_accuracy)
print("F1-score :", mnb_f1)

print("\nClassification Report - Multinomial Naive Bayes:")
print(
    classification_report(
        y_test,
        y_pred_mnb,
        target_names=train_data.target_names,
        zero_division=0
    )
)




# ---------------------------------------------------------
# 4. Bernoulli Naive Bayes
# ---------------------------------------------------------

# BernoulliNB works with binary features.
#
# binary=True converts word counts into:
#     1 -> word is present
#     0 -> word is absent

binary_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    binary=True
)

X_train_binary = binary_vectorizer.fit_transform(X_train_text)
X_test_binary = binary_vectorizer.transform(X_test_text)

# Create Bernoulli Naive Bayes classifier
bernoulli_nb = BernoulliNB(alpha=1.0)

# Train
bernoulli_nb.fit(X_train_binary, y_train)

# Predict
y_pred_bnb = bernoulli_nb.predict(X_test_binary)

# Evaluate
bnb_accuracy = accuracy_score(y_test, y_pred_bnb)
bnb_f1 = f1_score(
    y_test,
    y_pred_bnb,
    average="weighted"
)

print("\n========================================")
print("Bernoulli Naive Bayes")
print("========================================")
print("Accuracy :", bnb_accuracy)
print("F1-score :", bnb_f1)

print("\nClassification Report - Bernoulli Naive Bayes:")
print(
    classification_report(
        y_test,
        y_pred_bnb,
        target_names=train_data.target_names,
        zero_division=0
    )
)




# ---------------------------------------------------------
# 5. Compare Both Models
# ---------------------------------------------------------

results = pd.DataFrame({
    "Model": [
        "Multinomial Naive Bayes",
        "Bernoulli Naive Bayes"
    ],
    "Accuracy": [
        mnb_accuracy,
        bnb_accuracy
    ],
    "Weighted F1-score": [
        mnb_f1,
        bnb_f1
    ]
})

print("\n========================================")
print("MODEL COMPARISON")
print("========================================")
print(results.to_string(index=False))


# ---------------------------------------------------------
# 6. Identify the Better Model
# ---------------------------------------------------------

if mnb_f1 > bnb_f1:
    print("\nMultinomial Naive Bayes performs better based on F1-score.")
elif bnb_f1 > mnb_f1:
    print("\nBernoulli Naive Bayes performs better based on F1-score.")
else:
    print("\nBoth models have the same F1-score.")


# ---------------------------------------------------------
# 7. Optional: Confusion Matrices
# ---------------------------------------------------------

cm_mnb = confusion_matrix(y_test, y_pred_mnb)
cm_bnb = confusion_matrix(y_test, y_pred_bnb)

print("\nMultinomial NB Confusion Matrix:")
print(cm_mnb)

print("\nBernoulli NB Confusion Matrix:")
print(cm_bnb)


Number of training documents: 11314
Number of testing documents: 7532
Number of categories: 20

Categories:
0 : alt.atheism
1 : comp.graphics
2 : comp.os.ms-windows.misc
3 : comp.sys.ibm.pc.hardware
4 : comp.sys.mac.hardware
5 : comp.windows.x
6 : misc.forsale
7 : rec.autos
8 : rec.motorcycles
9 : rec.sport.baseball
10 : rec.sport.hockey
11 : sci.crypt
12 : sci.electronics
13 : sci.med
14 : sci.space
15 : soc.religion.christian
16 : talk.politics.guns
17 : talk.politics.mideast
18 : talk.politics.misc
19 : talk.religion.misc

Feature matrix shape:
Training: (11314, 147857)
Testing : (7532, 147857)

Multinomial Naive Bayes
Accuracy : 0.6428571428571429
F1-score : 0.6194633730146152

Classification Report - Multinomial Naive Bayes:
                          precision    recall  f1-score   support

             alt.atheism       0.63      0.33      0.44       319
           comp.graphics       0.53      0.75      0.62       389
 comp.os.ms-windows.misc       0.57      0.01      0.02      